# 01 — Music Feature Extraction

Extract symbolic music features from the MIDI files identified in notebook 00 using the
[**DIDONEproject/music_symbolic_features**](https://github.com/DIDONEproject/music_symbolic_features)
pipeline — which orchestrates **musif**, **music21**, and **jSymbolic** in an isolated
Python 3.10 environment.

**Pipeline**
1. One-time setup: `bash scripts/setup_didone.sh`
2. Run feature extraction: `bash scripts/extract_features.sh`
3. Load & merge the three extractor CSVs with `load_didone_features` / `merge_didone_features`
4. Post-process, inspect distributions & correlations
5. Merge with MSD metadata and save to `data/processed/music_features.parquet`

> **Fallback**: If the DIDONE pipeline has not been run yet, this notebook
> falls back to direct `musif` extraction via `batch_extract()`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.dataset_extraction import load_dataset
from data.music_feature_extraction import (
    # DIDONE pipeline (primary)
    load_didone_features,
    merge_didone_features,
    DIDONE_EXTRACTORS,
    # direct musif (fallback)
    extract_musif_features,
    postprocess_features,
    batch_extract,
    DEFAULT_FEATURES,
)

sns.set_theme(style='whitegrid')
print("Imports OK")


## 1 — Configuration

In [ ]:
REPO_ROOT        = pathlib.Path('..').resolve()
RAW_DIR          = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR    = REPO_ROOT / 'data' / 'processed'
INTERIM_DIR      = REPO_ROOT / 'data' / 'interim'

# ── input ─────────────────────────────────────────────────────────────────────
DATASET_PARQUET  = PROCESSED_DIR / 'lakh_msd_dataset.parquet'

# ── DIDONE pipeline paths ─────────────────────────────────────────────────────
DIDONE_OUTPUT    = INTERIM_DIR / 'didone_features'   # OUTPUT in settings.py
DATASET_NAME     = 'lmd_matched'                     # folder inside DIDONE_OUTPUT
MIDI_DIR         = RAW_DIR / 'lmd_matched'

# ── output ────────────────────────────────────────────────────────────────────
FEATURES_PARQUET = PROCESSED_DIR / 'music_features.parquet'
MERGED_PARQUET   = PROCESSED_DIR / 'lakh_msd_with_features.parquet'

# ── fallback (direct musif) settings ──────────────────────────────────────────
CHUNK_SIZE  = 500          # files per musif call
MAX_FILES   = None         # int for a quick test, None = all
FEATURES    = DEFAULT_FEATURES

# ── control flags ─────────────────────────────────────────────────────────────
# Set USE_DIDONE_PIPELINE = False to fall back to direct musif extraction
USE_DIDONE_PIPELINE = True

print(f"DIDONE output dir : {DIDONE_OUTPUT}")
print(f"Dataset name      : {DATASET_NAME}")
print(f"Features parquet  : {FEATURES_PARQUET}")
print(f"Use DIDONE        : {USE_DIDONE_PIPELINE}")


## 2 — DIDONE Pipeline Setup & Extraction

Run once to clone the repo, create a Python 3.10 venv, and download jSymbolic:

```bash
bash scripts/setup_didone.sh
```

Then extract features (this can take hours for the full LMD):

```bash
bash scripts/extract_features.sh
# or only one extractor:
# bash scripts/extract_features.sh --only musif
```

The cells below automate both steps via `subprocess`. Set `SKIP_SETUP=True` and
`SKIP_EXTRACTION=True` to skip if already done.


In [ ]:
import subprocess

SKIP_SETUP      = False   # set True after first successful run
SKIP_EXTRACTION = False   # set True after features are already extracted

# ── Setup ────────────────────────────────────────────────────────────────────
if not SKIP_SETUP:
    print("=" * 60)
    print(" Running scripts/setup_didone.sh …")
    print("=" * 60)
    setup_result = subprocess.run(
        ["bash", str(REPO_ROOT / "scripts" / "setup_didone.sh"),
         "--midi-dir", str(MIDI_DIR)],
        cwd=str(REPO_ROOT),
        capture_output=False,   # stream output live
    )
    if setup_result.returncode != 0:
        print(f"\n[ERROR] Setup failed with exit code {setup_result.returncode}")
    else:
        print("\n[OK] Setup complete.")
else:
    print("[SKIP] Setup skipped (SKIP_SETUP=True).")


In [ ]:
# ── Feature extraction ────────────────────────────────────────────────────────
if not SKIP_EXTRACTION and USE_DIDONE_PIPELINE:
    print("=" * 60)
    print(" Running scripts/extract_features.sh …")
    print(" (This may take a long time for large datasets)")
    print("=" * 60)
    extract_result = subprocess.run(
        ["bash", str(REPO_ROOT / "scripts" / "extract_features.sh")],
        cwd=str(REPO_ROOT),
        capture_output=False,
        env={
            **__import__("os").environ,
            "DATASET_NAME": DATASET_NAME,
        },
    )
    if extract_result.returncode != 0:
        print(f"\n[ERROR] Extraction failed with exit code {extract_result.returncode}")
        print("Check logs/ directory for details.")
    else:
        print("\n[OK] Extraction complete.")
elif SKIP_EXTRACTION:
    print("[SKIP] Extraction skipped (SKIP_EXTRACTION=True).")
else:
    print("[SKIP] DIDONE pipeline disabled (USE_DIDONE_PIPELINE=False).")


## 3 — Load Feature CSVs

Load each extractor's output and merge into a single wide DataFrame.
Each extractor's feature columns are prefixed (`musif__`, `music21__`, `jsymbolic__`)
to prevent naming collisions.


In [ ]:
if USE_DIDONE_PIPELINE:
    # Try to load all three extractor CSVs; silently skip missing ones
    features_df = merge_didone_features(
        output_dir=DIDONE_OUTPUT,
        dataset_name=DATASET_NAME,
        extractors=DIDONE_EXTRACTORS,
        extension="mid",
        how="outer",
        verbose=True,
    )

    if features_df.empty:
        print("\n[WARN] No DIDONE output found. Falling back to direct musif extraction.")
        USE_DIDONE_PIPELINE = False   # trigger fallback below
    else:
        print(f"\nLoaded DIDONE features: {features_df.shape}")
        print(features_df.head(3))


## 4 — Load MSD metadata


In [ ]:
meta_df = load_dataset(DATASET_PARQUET)
print(f"Loaded {len(meta_df):,} tracks")

midi_paths = meta_df['midi_path'].dropna().tolist()
if MAX_FILES:
    midi_paths = midi_paths[:MAX_FILES]
print(f"MIDI files to process: {len(midi_paths):,}")

## 4b — Fallback: direct musif extraction

Only runs if `USE_DIDONE_PIPELINE = False` or the DIDONE output is empty.


In [ ]:
if not USE_DIDONE_PIPELINE:
    raw_features_df = batch_extract(
        midi_paths   = midi_paths,
        out_parquet  = FEATURES_PARQUET,
        chunk_size   = CHUNK_SIZE,
        features     = FEATURES,
        parallel     = -1,
        overwrite    = False,   # set True to re-run from scratch
        verbose      = True,
    )
    print(f"\nRaw features shape: {raw_features_df.shape}")
else:
    print("[SKIP] Using DIDONE pipeline output (loaded above).")


## 5 — Post-process / clean feature DataFrame


In [ ]:
if not USE_DIDONE_PIPELINE:
    # Post-process the direct musif output
    features_df = postprocess_features(
        raw_features_df,
        config_yaml    = None,
        drop_threshold = 0.80,
        verbose        = True,
    )
else:
    # Drop columns with > 80 % NaN from the merged DIDONE frame
    thresh = int(len(features_df) * 0.20)
    before = features_df.shape[1]
    features_df = features_df.dropna(axis=1, thresh=max(thresh, 1))
    print(f"[postprocess] {before} → {features_df.shape[1]} columns "
          f"(dropped {before - features_df.shape[1]} high-NaN columns)")

print(f"\nFinal features shape: {features_df.shape}")
features_df.head(3)


## 6 — Feature exploration


In [ ]:
num_df   = features_df.select_dtypes(include='number')
nan_rate = num_df.isnull().mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: NaN rate histogram
axes[0].hist(nan_rate.values, bins=40, color='steelblue', edgecolor='white')
axes[0].set(title='NaN Rate Distribution (numeric columns)',
            xlabel='Fraction NaN', ylabel='# Columns')

# Right: distributions of a few candidate features (prefix-agnostic search)
_cands = ['Tempo_Mean', 'musif__Tempo_Mean', 'Ambitus_Ambitus', 'musif__Ambitus_Ambitus',
          'NoteDensity', 'jsymbolic__Number of Pitches',
          'music21__ambitus', 'tempo', 'Duration']
key_cols = [c for c in num_df.columns
            if any(kw.lower() in c.lower() for kw in ['tempo','ambitus','density','pitch'])][:6]

for col in key_cols:
    axes[1].hist(num_df[col].dropna(), bins=50, alpha=0.6, label=col.split('__')[-1])
axes[1].set(title='Selected Feature Distributions', xlabel='Value', ylabel='Count')
if key_cols:
    axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Total numeric features: {len(num_df.columns)}")
print(f"Median NaN rate: {nan_rate.median():.1%}")


In [ ]:
# Correlation heatmap — top 20 most-complete numeric features
top20 = nan_rate.index[:20].tolist()
corr  = num_df[top20].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.4, square=True)
plt.title('Feature Correlation Matrix (top-20 complete columns)')
plt.tight_layout()
plt.show()

## 6 — Merge with metadata & save final feature table

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save features alone
features_df.to_parquet(FEATURES_PARQUET, index=False)
print(f"Features saved → {FEATURES_PARQUET}  ({features_df.shape})")

# Merge with MSD metadata
if DATASET_PARQUET.exists():
    meta_df = load_dataset(DATASET_PARQUET)

    # Determine the join key: DIDONE uses FileName (stem), metadata uses midi_path
    if 'FileName' in features_df.columns:
        # FileName from DIDONE = file stem (e.g. "abc123.mid")
        # Build a lookup: file stem → midi_path
        meta_df['_fname'] = meta_df['midi_path'].apply(
            lambda p: pathlib.Path(p).name if pd.notna(p) else None
        )
        merged = features_df.merge(
            meta_df.drop(columns=['_fname'], errors='ignore').assign(
                _fname=meta_df['midi_path'].apply(
                    lambda p: pathlib.Path(p).name if pd.notna(p) else None
                )
            ),
            left_on='FileName', right_on='_fname', how='left'
        ).drop(columns=['_fname'], errors='ignore')
    else:
        merged = features_df.merge(meta_df, on='midi_path', how='left')

    merged.to_parquet(MERGED_PARQUET, index=False)
    print(f"Merged  saved → {MERGED_PARQUET}  ({merged.shape})")
    merged.head(2)
else:
    print(f"[WARN] Metadata not found: {DATASET_PARQUET}")
    print("       Run notebook 00 first to generate the metadata Parquet.")
    features_df.head(2)
